# Student Performance Analysis
## Pandas: NaN Handling, GroupBy, Merge/Join, Pivot Table

This notebook analyses a synthetic university dataset across three related tables:
- **students** — demographic and academic info
- **courses** — course catalog with categories and credits
- **grades** — exam scores per student per course

**Topics covered:**
- NaN detection and filling strategies
- Derived columns (weighted score, pass/fail)
- Merging multiple tables (left join)
- GroupBy aggregations
- Pivot tables
- Concat and duplicate checks

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Pandas version:", pd.__version__)

Pandas version: 3.0.2


## 2. Mini Example — Before the Real Data

Two small DataFrames to practice the core operations cleanly.

### 2.1 Build Two Tables

In [2]:
employees = pd.DataFrame({
    "emp_id"    : [1, 2, 3, 4],
    "name"      : ["Alice", "Bob", "Carol", "Diana"],
    "dept"      : ["DS", "DS", "Engineering", "DS"],
    "score"     : [88.5, np.nan, 72.0, 91.3]
})

departments = pd.DataFrame({
    "dept"    : ["DS", "Engineering"],
    "faculty" : ["Science", "Engineering"]
})

employees, departments

(   emp_id   name         dept  score
 0       1  Alice           DS   88.5
 1       2    Bob           DS    NaN
 2       3  Carol  Engineering   72.0
 3       4  Diana           DS   91.3,
           dept      faculty
 0           DS      Science
 1  Engineering  Engineering)

### 2.2 Detect and Fill NaN

In [3]:
print(employees.isna())

# Fill missing score with column mean
employees["score"] = employees["score"].fillna(employees["score"].mean())
employees

   emp_id   name   dept  score
0   False  False  False  False
1   False  False  False   True
2   False  False  False  False
3   False  False  False  False


,emp_id,name,dept,score
0,1,Alice,DS,88.500000
1,2,Bob,DS,83.933333
2,3,Carol,Engineering,72.000000
3,4,Diana,DS,91.300000


### 2.3 Merge Tables

In [4]:
merged = employees.merge(departments, on="dept", how="left")
merged

,emp_id,name,dept,score,faculty
0,1,Alice,DS,88.500000,Science
1,2,Bob,DS,83.933333,Science
2,3,Carol,Engineering,72.000000,Engineering
3,4,Diana,DS,91.300000,Science


### 2.4 GroupBy Summary

In [5]:
merged.groupby("dept")["score"].agg(["count", "mean"]).reset_index()

,dept,count,mean
0,DS,3,87.911111
1,Engineering,1,72.000000


## 3. Load Real CSV Files

Three related tables — make sure all CSV files are in the same folder as this notebook.

In [6]:
students = pd.read_csv("ybs_ogrenciler_ascii.csv")
courses  = pd.read_csv("ybs_dersler_ascii.csv")
grades   = pd.read_csv("ybs_notlar_ascii.csv")

print("Students :", students.shape)
print("Courses  :", courses.shape)
print("Grades   :", grades.shape)

Students : (220, 8)
Courses  : (10, 4)
Grades   : (900, 6)


In [7]:
students.head()

,ogrenci_id,ad,soyad,sehir,bolum,sinif,gpa,email
0,200001,Burak,Celik,Samsun,YBS,4,1.76,burak.celik33@example.com
1,200002,Ece,Yildiz,Adana,Endustri,4,2.48,ece.yildiz54@example.com
2,200003,Mert,Karaca,Konya,Ekonomi,2,2.28,mert.karaca44@example.com
3,200004,Can,Tas,Samsun,YBS,3,3.13,can.tas93@example.com
4,200005,Sibel,Kurt,Kocaeli,YBS,2,2.98,sibel.kurt31@example.com


In [8]:
courses.head()

,ders_kodu,ders_adi,kategori,kredi
0,DS101,VeriBilimi,Data,3
1,DB201,VeriTabani,Data,3
2,PY101,PythonProgramlama,Tech,2
3,ST201,Istatistik,Math,2
4,BA301,IsAnalitigi,Business,3


In [9]:
grades.head()

,kayit_id,ogrenci_id,ders_kodu,donem,vize,final
0,500001,200189,DS101,2026-Spring,84.8,62.7
1,500002,200160,AI201,2025-Fall,65.7,79.2
2,500003,200183,MK201,2025-Fall,100.0,51.0
3,500004,200202,FN201,2026-Spring,67.4,NaN
4,500005,200047,AI201,2025-Fall,62.6,97.5


## 4. Explore the Data

In [10]:
students.info()

<class 'pandas.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ogrenci_id  220 non-null    int64  
 1   ad          220 non-null    str    
 2   soyad       220 non-null    str    
 3   sehir       220 non-null    str    
 4   bolum       220 non-null    str    
 5   sinif       220 non-null    int64  
 6   gpa         205 non-null    float64
 7   email       204 non-null    str    
dtypes: float64(1), int64(2), str(5)
memory usage: 13.9 KB


In [11]:
grades.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   kayit_id    900 non-null    int64  
 1   ogrenci_id  900 non-null    int64  
 2   ders_kodu   900 non-null    str    
 3   donem       900 non-null    str    
 4   vize        845 non-null    float64
 5   final       832 non-null    float64
dtypes: float64(2), int64(2), str(2)
memory usage: 42.3 KB


## 5. NaN Handling

### 5.1 Count Missing Values

In [12]:
print("=== Students ===")
print(students.isna().sum())

print("\n=== Grades ===")
print(grades.isna().sum())

=== Students ===
ogrenci_id     0
ad             0
soyad          0
sehir          0
bolum          0
sinif          0
gpa           15
email         16
dtype: int64

=== Grades ===
kayit_id       0
ogrenci_id     0
ders_kodu      0
donem          0
vize          55
final         68
dtype: int64


### 5.2 Fill Missing GPA with Column Mean

In [13]:
students["gpa"] = students["gpa"].fillna(students["gpa"].mean())
print(students.isna().sum())

ogrenci_id     0
ad             0
soyad          0
sehir          0
bolum          0
sinif          0
gpa            0
email         16
dtype: int64


### 5.3 Fill Missing Exam Scores with 0

A missing score means the student did not sit the exam.

In [14]:
grades[["vize", "final"]] = grades[["vize", "final"]].fillna(0)
print(grades.isna().sum())

kayit_id      0
ogrenci_id    0
ders_kodu     0
donem         0
vize          0
final         0
dtype: int64


## 6. Derived Columns: Weighted Score and Pass/Fail

Formula: `total = 0.4 × midterm + 0.6 × final`  
Pass threshold: total ≥ 60

In [15]:
grades["total_score"] = 0.4 * grades["vize"] + 0.6 * grades["final"]
grades["result"]      = np.where(grades["total_score"] >= 60, "Pass", "Fail")
grades.head()

,kayit_id,ogrenci_id,ders_kodu,donem,vize,final,total_score,result
0,500001,200189,DS101,2026-Spring,84.8,62.7,71.54,Pass
1,500002,200160,AI201,2025-Fall,65.7,79.2,73.80,Pass
2,500003,200183,MK201,2025-Fall,100.0,51.0,70.60,Pass
3,500004,200202,FN201,2026-Spring,67.4,0.0,26.96,Fail
4,500005,200047,AI201,2025-Fall,62.6,97.5,83.54,Pass


## 7. Merge / Join — Combine All Three Tables

**Step 1:** grades + students (on student ID)  
**Step 2:** result + courses (on course code)

In [16]:
df = grades.merge(students, on="ogrenci_id", how="left")
df = df.merge(courses,  on="ders_kodu",  how="left")
df.head()

,kayit_id,ogrenci_id,ders_kodu,donem,vize,final,total_score,result,ad,soyad,sehir,bolum,sinif,gpa,email,ders_adi,kategori,kredi
0,500001,200189,DS101,2026-Spring,84.8,62.7,71.54,Pass,Ece,Gunes,Istanbul,Isletme,3,3.270000,ece.gunes59@example.com,VeriBilimi,Data,3
1,500002,200160,AI201,2025-Fall,65.7,79.2,73.80,Pass,Sena,Ozdemir,Eskisehir,Isletme,4,1.970000,NaN,YapayZekaTemelleri,Tech,3
2,500003,200183,MK201,2025-Fall,100.0,51.0,70.60,Pass,Kerem,Arslan,Bursa,Bilgisayar,4,2.860000,kerem.arslan41@example.com,PazarlamaAnalizi,Business,4
3,500004,200202,FN201,2026-Spring,67.4,0.0,26.96,Fail,Ozan,Yildiz,Eskisehir,Endustri,4,2.490000,ozan.yildiz52@example.com,FinansAnalizi,Business,3
4,500005,200047,AI201,2025-Fall,62.6,97.5,83.54,Pass,Sena,Aydin,Bursa,Isletme,2,2.690244,sena.aydin89@example.com,YapayZekaTemelleri,Tech,3


In [17]:
print("Merged shape:", df.shape)
print("Columns:", list(df.columns))

Merged shape: (900, 18)
Columns: ['kayit_id', 'ogrenci_id', 'ders_kodu', 'donem', 'vize', 'final', 'total_score', 'result', 'ad', 'soyad', 'sehir', 'bolum', 'sinif', 'gpa', 'email', 'ders_adi', 'kategori', 'kredi']


## 8. GroupBy — Summary Reports

### 8.1 Average Score by Department

In [18]:
dept_summary = (
    df.groupby("bolum")["total_score"]
    .agg(count="count", mean="mean", min="min", max="max")
    .reset_index()
    .sort_values("mean", ascending=False)
)
dept_summary

,bolum,count,mean,min,max
1,Ekonomi,56,64.812500,16.20,95.44
3,Isletme,207,64.480773,0.00,94.78
2,Endustri,177,64.414576,10.32,94.76
4,YBS,398,61.950553,7.14,99.16
0,Bilgisayar,62,61.854516,18.32,95.04


### 8.2 Pass Rate by Course

In [19]:
course_report = df.groupby("ders_kodu").agg(
    total_students = ("kayit_id",    "count"),
    avg_score      = ("total_score", "mean"),
    pass_count     = ("result",      lambda x: (x == "Pass").sum())
).reset_index()

course_report["pass_rate"] = course_report["pass_count"] / course_report["total_students"]
course_report.sort_values("pass_rate", ascending=False).head(10)

,ders_kodu,total_students,avg_score,pass_count,pass_rate
8,ST201,85,63.906824,61,0.717647
4,DS101,98,65.584898,67,0.683673
5,FN201,91,64.726374,62,0.681319
3,DB201,82,64.048537,55,0.670732
9,WEB101,93,64.059140,62,0.666667
0,AI201,95,61.253684,62,0.652632
7,PY101,91,63.674945,58,0.637363
2,BI201,94,62.412553,56,0.595745
6,MK201,94,59.406809,54,0.574468
1,BA301,77,62.937403,44,0.571429


### 8.3 Average Score by Year (sinif)

In [20]:
df.groupby("sinif")["total_score"].mean().reset_index().sort_values("sinif")

,sinif,total_score
0,1,62.255385
1,2,64.055385
2,3,63.263073
3,4,62.509309


## 9. Pivot Table — Department × Course Category

Shows the mean total score for each department across course categories.

In [21]:
pivot1 = pd.pivot_table(
    df,
    values   = "total_score",
    index    = "bolum",
    columns  = "kategori",
    aggfunc  = "mean"
)
pivot1.round(2)

kategori,Business,Data,Math,Tech
bolum,,,,
Bilgisayar,58.26,72.59,67.81,59.22
Ekonomi,65.64,61.57,68.61,64.81
Endustri,65.00,68.51,56.27,63.42
Isletme,64.13,66.31,67.75,62.35
YBS,60.38,62.33,62.36,63.64


### 9.1 Pivot Table — Department × Course Code

In [22]:
pivot2 = pd.pivot_table(
    df,
    values  = "total_score",
    index   = "bolum",
    columns = "ders_kodu",
    aggfunc = "mean"
)
pivot2.round(2)

ders_kodu,AI201,BA301,BI201,DB201,DS101,FN201,MK201,PY101,ST201,WEB101
bolum,,,,,,,,,,
Bilgisayar,52.94,50.45,57.29,65.75,75.53,64.82,62.93,57.45,67.81,67.62
Ekonomi,52.70,74.16,69.13,67.10,57.89,52.24,64.62,77.09,68.61,67.70
Endustri,67.09,67.49,62.33,69.83,67.34,66.91,65.38,62.85,56.27,60.69
Isletme,58.18,64.12,64.22,68.19,64.67,66.59,59.32,66.62,67.75,62.64
YBS,63.05,61.92,61.18,58.97,65.37,63.79,56.26,62.61,62.36,65.44


## 10. Concat — Adding New Records

Simulate new student registrations arriving after the original dataset.

In [23]:
new_students = pd.DataFrame({
    "ogrenci_id" : [999001, 999002],
    "ad"         : ["Lena",   "Max"],
    "soyad"      : ["Smith",  "Brown"],
    "sehir"      : ["Ankara", "Istanbul"],
    "bolum"      : ["YBS",    "Ekonomi"],
    "sinif"      : [2,        3],
    "gpa"        : [3.10,     2.75],
    "email"      : ["lena@example.com", "max@example.com"]
})

students_updated = pd.concat([students, new_students], ignore_index=True)
students_updated.tail()

,ogrenci_id,ad,soyad,sehir,bolum,sinif,gpa,email
217,200218,Selin,Gunes,Antalya,Endustri,3,2.51,selin.gunes93@example.com
218,200219,Can,Erdogan,Istanbul,YBS,3,2.97,can.erdogan52@example.com
219,200220,Selin,Celik,Ankara,YBS,2,1.78,selin.celik63@example.com
220,999001,Lena,Smith,Ankara,YBS,2,3.10,lena@example.com
221,999002,Max,Brown,Istanbul,Ekonomi,3,2.75,max@example.com


## 11. Duplicate Check

Verify no student ID appears more than once in the updated table.

In [24]:
duplicate_count = students_updated["ogrenci_id"].duplicated().sum()
print("Duplicate student IDs:", duplicate_count)

Duplicate student IDs: 0


## Summary

| Topic | Method used |
|---|---|
| Missing values | `isna()`, `fillna()` |
| Derived columns | arithmetic + `np.where` |
| Table joins | `merge(..., how="left")` |
| Aggregation | `groupby().agg()` |
| Cross-tab report | `pd.pivot_table()` |
| Stacking rows | `pd.concat()` |
| Deduplication | `duplicated().sum()` |